<a href="https://colab.research.google.com/github/Garcchi-Prog/Lab1EDDII/blob/main/Lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
## Imports ##

from typing import Optional, Tuple
from graphviz import Digraph
from IPython.display import SVG
import csv

import pandas as pd

df = pd.read_csv('dataset_courses_with_reviews.csv', dtype=str)
df = df.fillna('0')  # Reemplaza vacíos con 0
indice_csv = {row['id']: row.tolist() for _, row in df.iterrows()}

In [25]:
## Clase Nodo ##

class Node:

    def __init__(self, data: list[str]) -> None:
        self.data = data
        self.satis = self.satisfaccion(self.data[3], self.data[11], self.data[12], self.data[13], self.data[4])
        self.izquierda: Optional["Node"] = None
        self.derecha: Optional["Node"] = None

    def satisfaccion(self, rating, positive_reviews, negative_reviews, neutral_reviews, num_reviews):
        try:
            rating = float(rating)
            positive_reviews = float(positive_reviews)
            negative_reviews = float(negative_reviews)
            neutral_reviews = float(neutral_reviews)
            num_reviews = float(num_reviews)

            if num_reviews != 0:
                val = rating * 0.7 + ((5 * positive_reviews + negative_reviews + 3 * neutral_reviews) / num_reviews) * 0.3
                return round(val, 5)
            else:
                return 0.0
        except ValueError:
            return 0.0

In [26]:
## Clase Arbol ##

class Tree:
    def __init__(self, raiz: Optional["Node"] = None) -> None:
        self.raiz = raiz
    
    def dibujar(self):
        dot = Digraph(comment='Árbol AVL')
        dot.attr(rankdir='TB')
        def agregar_nodos(nodo):
            if nodo is None:
                return
            titulo = nodo.data[1][:20] + "..." if len(nodo.data[1]) > 20 else nodo.data[1]
            label = f"{nodo.data[0]}\n{titulo}\nSatis: {nodo.satis:.5f}"
            dot.node(str(id(nodo)), label, shape='box')
            if nodo.izquierda:
                dot.edge(str(id(nodo)), str(id(nodo.izquierda)), label='I')
                agregar_nodos(nodo.izquierda)
            if nodo.derecha:
                dot.edge(str(id(nodo)), str(id(nodo.derecha)), label='D')
                agregar_nodos(nodo.derecha)
        agregar_nodos(self.raiz)
        return dot

        

    def add_node(self, node_id: str) -> str:
        fila = indice_csv.get(node_id)

        if fila is None:
            return "La id del curso no existe en el dataset"

        newNode = Node(fila)

        if self.raiz is None:
            self.raiz = newNode
            return "Nodo raíz añadido exitosamente!"

        current = self.raiz
        parent = None

        while current is not None:
            parent = current
            if newNode.satis > current.satis:
                current = current.derecha
            elif newNode.satis < current.satis:
                current = current.izquierda
            else:
                return "Error! No pueden existir 2 nodos con el mismo valor de satisfacción!"

        if newNode.satis > parent.satis:
            parent.derecha = newNode
        else:
            parent.izquierda = newNode

        return "Nodo añadido exitosamente!"

    def search_node(self, dataSearch: str, option: str) -> Optional["Node"]:
        # option: "id" para buscar por ID, "satis" para buscar por satisfacción

        if option == "id":
            return self._search_by_id(dataSearch, self.raiz)

        elif option == "satis":
            current = self.raiz
            satis_buscada = float(dataSearch)
            while current is not None:
                if satis_buscada == current.satis:
                    return current
                elif satis_buscada > current.satis:
                    current = current.derecha
                else:
                    current = current.izquierda
            return None

        return None

    def _search_by_id(self, node_id: str, nodo: Optional["Node"]) -> Optional["Node"]:
        # Búsqueda recursiva por ID
        if nodo is None:
            return None
        if nodo.data[0] == node_id:
            return nodo
        izq = self._search_by_id(node_id, nodo.izquierda)
        if izq:
            return izq
        return self._search_by_id(node_id, nodo.derecha)

    def delete_node(self, dataSearch: str, option: str) -> str:
        searchedNode = self.search_node(dataSearch, option)

        if searchedNode is None:
            return "El nodo no fue encontrado, revise la información proporcionada."

        self.raiz = self._delete_recursivo(self.raiz, searchedNode.satis)
        return "Nodo eliminado exitosamente!"

    def _delete_recursivo(self, nodo: Optional["Node"], satis: float) -> Optional["Node"]:
        if nodo is None:
            return None

        if satis < nodo.satis:
            nodo.izquierda = self._delete_recursivo(nodo.izquierda, satis)
        elif satis > nodo.satis:
            nodo.derecha = self._delete_recursivo(nodo.derecha, satis)
        else:
            # Nodo encontrado — 3 casos
            if nodo.izquierda is None:
                return nodo.derecha
            elif nodo.derecha is None:
                return nodo.izquierda
            else:
                # Tiene 2 hijos: reemplazar con el sucesor (mínimo del subárbol derecho)
                sucesor = self._minimo(nodo.derecha)
                nodo.data  = sucesor.data
                nodo.satis = sucesor.satis
                nodo.derecha = self._delete_recursivo(nodo.derecha, sucesor.satis)

        return equilibrar(nodo)

    def _minimo(self, nodo: "Node") -> "Node":
        while nodo.izquierda is not None:
            nodo = nodo.izquierda
        return nodo
    
## Codigo AVL ## 

def rotacion_simple_derecha(self, nodo):
    #La raiz de la rotacion es el hijo izquierdo del nodo desbalanceado
    n_raiz = nodo.izquierda

    #La informacion de la derecha de la nueva raiz pasa a ser la izquierda del nodo
    nodo.izquierda = n_raiz.derecha

    #El nodo desbalanceado pasa a ser la derecha de la nueva raiz
    n_raiz.derecha = nodo

    return n_raiz

def rotacion_simple_izquierda(self, nodo):
    #La raiz de la rotacion es el hijo derecho del nodo desbalanceado
    n_raiz = nodo.derecha

    #La informacion de la izquierda de la nueva raiz pasa a ser la derecha del nodo
    nodo.derecha = n_raiz.izquierda

    #El nodo desbalanceado pasa a ser la izquierda de la nueva raiz
    n_raiz.izquierda = nodo

    return n_raiz

def rotacion_doble_izquierda_derecha(self, nodo):
    #Se hace una rotacion simple a la izquierda del hijo izquierdo del nodo desbalanceado
    nodo.izquierda = rotacion_simple_izquierda(nodo.izquierda)

    #Luego se hace una rotacion simple a la derecha del nodo desbalanceado
    n_raiz= rotacion_simple_derecha(nodo)

    return n_raiz

def rotacion_doble_derecha_izquierda(self, nodo):
    #Se hace una rotacion simple a la derecha del hijo derecho del nodo desbalanceado
    nodo.derecha = rotacion_simple_derecha(nodo.derecha)

    #Luego se hace una rotacion simple a la izquierda del nodo desbalanceado
    n_raiz= rotacion_simple_izquierda(nodo)

    return n_raiz

def equilibrar(self, nodo):
    if nodo is None:
        return nodo

    #Se hace el calculo del equilibrio del nodo 
    equilibrio = obtener_equilibrio(nodo)

    #Caso de desbalanceo (2)
    if equilibrio > 1:
        #Si el hijo izquierdo tiene equilibrio 0 o 1 solamente se hace simple
        if obtener_equilibrio(nodo.izquierda) >= 0:
            return rotacion_simple_derecha(nodo)

        #Si el hijo izquierdo tiene equilibrio -1 se aplica rotacion doble
        else:
            return rotacion_doble_izquierda_derecha(nodo)

    #Caso de desbalanceo (-2)
    if equilibrio < -1:
        #Si el hijo derecho tiene equilibrio 0 o -1 solamente se hace simple
        if obtener_equilibrio(nodo.derecha) <= 0:
            return rotacion_simple_izquierda(nodo)

        #Si el hijo derecho tiene equilibrio 1 se aplica rotacion doble
        else:
            return rotacion_doble_derecha_izquierda(nodo)

    return nodo

def obtener_equilibrio(self, nodo):
    if nodo is None:
        return 0

    #Se hace un llamado para calcular la altura de los hijos del nodo e inmediatamente se calcula el equilibrio 
    return altura(nodo.derecha) - altura(nodo.izquierda)

def altura(self, nodo):
    if nodo is None:
        return 0

    #Se aplica recursion para calcular la altura de los hijos del nodo
    return 1 + max(altura(nodo.izquierda), altura(nodo.derecha))

#CODIGO DE LAS OPERACIONES PARA BUSCAR PADRE, ABUELO Y TIO (RECURSIVO)

def BuscarPadre( raiz, nodo):
    if raiz is None:
        return None
    
    # Verificamos si  la raiz es el padre del nodo buscado tanto en la izquierda como en la derecha
    if (raiz.izquierda is  nodo) or (raiz.derecha is nodo):
        return raiz
    
    #Se aplica recursion  para buscar el padre en el subarbol izquierdo 
    padre_izquierda = BuscarPadre(raiz.izquierda, nodo)
    if padre_izquierda is not None:
        return padre_izquierda
    
    #Se aplica recursion para buscar el padre en el subarbol derecho
    padre_derecha = BuscarPadre(raiz.derecha, nodo)
    if padre_derecha is not None:
        return padre_derecha

    #Si no se encuentra el nodo en ninguno de los subarboles, se devuelve none
    return None

def BuscarAbuelo ( raiz, nodo):
    if raiz is None:
        return None
    
    #Se busca el padre del nodo dado
    padre = BuscarPadre(raiz, nodo)
    if padre is None:
        return None
    
    #Se busca el abuelo utilizando la funcion BuscarPadre, solamente que el nodo a buscar sera el del padre 
    abuelo=BuscarPadre(raiz, padre)
    return abuelo

def BuscarTio ( raiz, nodo):
    if raiz is None:
        return None

    #Se busca al padre para tener una referencia para buscar el abuelo
    padre = BuscarPadre(raiz, nodo)
    #Si el padre no existe, no se puede encontrar al tio
    if padre is None:
        return None
    
    #Se busca el abuelo para tener una referencia para localizar al tio
    abuelo = BuscarPadre(raiz, padre)
    #Si el abuelo no existe, no se puede encontrar al tio
    if abuelo is None:
        return None
    
    #Caso 1: El hijo izquierdo del abuelo es el padre, por lo tanto el hijo derecho sera el tio
    if abuelo.izquierda is padre:
        return abuelo.derecha
    #Caso 2: El hijo derecho del abuelo es el padre, por lo tanto el hijo izquierdo sera el tio
    elif abuelo.derecha is padre:
        return abuelo.izquierda
    #Caso 3: El padre no es hijo del abuelo, por lo cual se retorna None
    else:
        return None
    
#Codigo para obtener el nivel de un nodo

def obtener_nivel(raiz, nodo):
    if raiz is None:
        return -1

    if raiz is nodo:
        return 0

    nivel_izquierda = obtener_nivel(raiz.izquierda, nodo)
    if nivel_izquierda != -1:
        return nivel_izquierda + 1
        
    nivel_derecha = obtener_nivel(raiz.derecha, nodo)
    if nivel_derecha != -1:
        return nivel_derecha + 1

In [27]:
def add_node(self, node_id: str) -> str:
    fila = indice_csv.get(node_id)
    
    if fila is None:
        return "La id del curso no existe en el dataset"
    
    newNode = Node(fila)
    
    if self.raiz is None:
        self.raiz = newNode
        return "Nodo raíz añadido exitosamente!"
    
    current = self.raiz
    parent = None

    while current is not None:
        parent = current
        if newNode.satis > current.satis:
            current = current.derecha
        elif newNode.satis < current.satis:
            current = current.izquierda
        else:
            return "Error! No pueden existir 2 nodos con el mismo valor de satisfacción!"

    if newNode.satis > parent.satis:
        parent.derecha = newNode
    else:
        parent.izquierda = newNode

    return "Nodo añadido exitosamente!"

In [28]:
def search_node(self, dataSearch: str, option: str) -> Optional["Node"]:
    # option: "id" para buscar por ID, "satis" para buscar por satisfacción
    
    current = self.raiz

    if option == "id":
        # Búsqueda recorriendo todo el árbol (el ID no es la métrica de orden)
        if current is None:
            return None
        if current.data[0] == dataSearch:
            return current
        izq = self.search_node_id(dataSearch, current.izquierda)
        if izq:
            return izq
        return self.search_node_id(dataSearch, current.derecha)

    elif option == "satis":
        # Búsqueda eficiente aprovechando el orden AVL
        satis_buscada = float(dataSearch)
        while current is not None:
            if satis_buscada == current.satis:
                return current
            elif satis_buscada > current.satis:
                current = current.derecha
            else:
                current = current.izquierda
        return None

def search_node_id(self, node_id: str, nodo: Optional["Node"]) -> Optional["Node"]:
    # Auxiliar recursiva para buscar por ID
    if nodo is None:
        return None
    if nodo.data[0] == node_id:
        return nodo
    izq = self.search_node_id(node_id, nodo.izquierda)
    if izq:
        return izq
    return self.search_node_id(node_id, nodo.derecha)

In [29]:
def insertar_nodos(mi_arbol, lista_ids):
    #Inserta una lista de IDs en el arbol Ejemplo: insertar_nodos(mi_arbol, ["123", "456", "789"])
    resultados = []
    for node_id in lista_ids:
        if node_id not in indice_csv:
            resultados.append(f"ID {node_id}: no existe en dataset")
            continue
            
        resultado = mi_arbol.add_node(node_id)
        fila = indice_csv[node_id]
        resultados.append(f"ID {node_id} ({fila[1][:30]}...): {resultado}")
    
    return "\n".join(resultados)

# Ejemplo de uso (modifica esta lista con los IDs que quieras insertar):
mi_arbol = Tree()
ids_a_insertar = list(indice_csv.keys())[:3]  # Primeros 3 del dataset
print(insertar_nodos(mi_arbol, ids_a_insertar))

ID 567828 (The Complete Python Bootcamp F...): Nodo raíz añadido exitosamente!
ID 1565838 (The Complete 2023 Web Developm...): Nodo añadido exitosamente!
ID 625204 (The Web Developer Bootcamp 202...): Nodo añadido exitosamente!


In [30]:
def buscar_nodo(mi_arbol, valor, tipo="id"):
    """
    Busca un nodo por ID o por satisfaccion.
    tipo: "id" o "satis"
    Ejemplo: buscar_nodo(mi_arbol, "123", "id")
             buscar_nodo(mi_arbol, "4.52345", "satis")
    """
    nodo = mi_arbol.search_node(valor, tipo)
    
    if nodo is None:
        return f"No se encontro nodo con {tipo} = {valor}"
    
    return {
        "id": nodo.data[0],
        "titulo": nodo.data[1],
        "satisfaccion": nodo.satis,
        "rating": nodo.data[3],
        "num_reviews": nodo.data[4]
    }

# Ejemplo de busqueda por ID (cambia el ID por el que quieras buscar):
id_buscar = list(indice_csv.keys())[0]
print("Busqueda por ID:")
print(buscar_nodo(mi_arbol, id_buscar, "id"))

# Ejemplo de busqueda por satisfaccion (usa el valor de satis de la raiz o cualquier otro):
if mi_arbol.raiz:
    satis_buscar = str(mi_arbol.raiz.satis)
    print("\nBusqueda por satisfaccion:")
    print(buscar_nodo(mi_arbol, satis_buscar, "satis"))

Busqueda por ID:
{'id': '567828', 'titulo': 'The Complete Python Bootcamp From Zero to Hero in Python', 'satisfaccion': 4.65648, 'rating': '4.5927815', 'num_reviews': '452973'}

Busqueda por satisfaccion:
{'id': '567828', 'titulo': 'The Complete Python Bootcamp From Zero to Hero in Python', 'satisfaccion': 4.65648, 'rating': '4.5927815', 'num_reviews': '452973'}


In [31]:
def verificar_parentesco(mi_arbol, node_id):
    """
    Muestra el parentesco de un nodo dado su ID.
    Ejemplo: verificar_parentesco(mi_arbol, "123")
    """
    nodo = mi_arbol.search_node(node_id, "id")
    
    if nodo is None:
        return f"Nodo con ID {node_id} no encontrado"
    
    padre = BuscarPadre(mi_arbol.raiz, nodo)
    abuelo = BuscarAbuelo(mi_arbol.raiz, nodo)
    tio = BuscarTio(mi_arbol.raiz, nodo)
    nivel = obtener_nivel(mi_arbol.raiz, nodo)
    
    info = {
        "nodo_id": nodo.data[0],
        "nodo_titulo": nodo.data[1],
        "nodo_satis": nodo.satis,
        "nivel": nivel,
        "padre": padre.data[1] if padre else "No tiene",
        "abuelo": abuelo.data[1] if abuelo else "No tiene",
        "tio": tio.data[1] if tio else "No tiene"
    }
    
    return f"""
    Nodo   : {info['nodo_titulo']} (ID: {info['nodo_id']})
    Nivel  : {info['nivel']}
    Padre  : {info['padre']}
    Abuelo : {info['abuelo']}
    Tio    : {info['tio']}
    """

# Ejemplo de uso (cambia el ID por el que quieras consultar):
if len(list(indice_csv.keys())) >= 3:
    tercer_id = list(indice_csv.keys())[2]
    print(verificar_parentesco(mi_arbol, tercer_id))


    Nodo   : The Web Developer Bootcamp 2023 (ID: 625204)
    Nivel  : 2
    Padre  : The Complete 2023 Web Development Bootcamp
    Abuelo : The Complete Python Bootcamp From Zero to Hero in Python
    Tio    : No tiene
    


In [32]:
#Aspecto visual
import ipywidgets as widgets
from IPython.display import display, clear_output, SVG
import datetime

# Instancia global del árbol (vacío al inicio)
arbol = Tree()

# ---------- Widgets para operaciones básicas ----------
id_insert = widgets.Text(description="ID:", placeholder="Ej: 567828")
btn_insert = widgets.Button(description="Insertar", button_style='primary')

delete_val = widgets.Text(description="Valor:", placeholder="ID o satisfacción")
delete_opt = widgets.Dropdown(options=['id', 'satis'], description="Buscar por:")
btn_delete = widgets.Button(description="Eliminar", button_style='danger')

search_val = widgets.Text(description="Valor:", placeholder="ID o satisfacción")
search_opt = widgets.Dropdown(options=['id', 'satis'], description="Buscar por:")
btn_search = widgets.Button(description="Buscar")

# ---------- Widgets para criterios ----------
btn_crit_a = widgets.Button(description="a) Positivas > Negativas+Neutras")

date_picker = widgets.DatePicker(description="Fecha (YYYY-MM-DD):")
btn_crit_b = widgets.Button(description="b) Creados después de fecha")

rango_clases = widgets.IntRangeSlider(
    value=[0, 100], min=0, max=500, step=10,
    description="Rango de clases:", continuous_update=False
)
btn_crit_c = widgets.Button(description="c) Clases en rango")

tipo_resena = widgets.Dropdown(options=['positive', 'negative', 'neutral'], description="Tipo reseña:")
btn_crit_d = widgets.Button(description="d) Reseñas > promedio total")

# ---------- Recorrido por niveles ----------
btn_niveles = widgets.Button(description="Mostrar recorrido por niveles (recursivo)")

# ---------- Áreas de salida ----------
output_mensajes = widgets.Output()
output_info = widgets.Output()      # Para mostrar información del nodo buscado
output_arbol = widgets.Output()     # Para la imagen del árbol

# ---------- Layout (dos columnas) ----------
left_panel = widgets.VBox([
    widgets.HTML("<h3>Operaciones básicas</h3>"),
    widgets.HBox([id_insert, btn_insert]),
    widgets.HBox([delete_val, delete_opt, btn_delete]),
    widgets.HBox([search_val, search_opt, btn_search]),
    widgets.HTML("<h3>Búsquedas por criterios</h3>"),
    btn_crit_a,
    widgets.HBox([date_picker, btn_crit_b]),
    widgets.HBox([rango_clases, btn_crit_c]),
    widgets.HBox([tipo_resena, btn_crit_d]),
    btn_niveles,
    output_mensajes,
    output_info
], layout=widgets.Layout(width='40%', padding='10px'))

right_panel = widgets.VBox([
    widgets.HTML("<h3>Árbol AVL</h3>"),
    output_arbol
], layout=widgets.Layout(width='60%', padding='10px'))

ui = widgets.HBox([left_panel, right_panel])

# ---------- Funciones auxiliares ----------
def actualizar_arbol():
    """Refresca la imagen del árbol en el panel derecho."""
    output_arbol.clear_output()
    with output_arbol:
        if arbol.raiz is None:
            display(widgets.HTML("<i>Árbol vacío</i>"))
        else:
            svg = arbol.dibujar().pipe(format='svg')
            display(SVG(svg))

def mostrar_mensaje(texto, error=False):
    with output_mensajes:
        clear_output()
        color = "red" if error else "green"
        display(widgets.HTML(f"<p style='color:{color}'>{texto}</p>"))

# ---------- Callbacks ----------
def on_insert(b):
    id_ = id_insert.value.strip()
    if not id_:
        mostrar_mensaje("Ingrese un ID", error=True)
        return
    resultado = arbol.add_node(id_)
    mostrar_mensaje(resultado)
    actualizar_arbol()

def on_delete(b):
    valor = delete_val.value.strip()
    if not valor:
        mostrar_mensaje("Ingrese un valor", error=True)
        return
    resultado = arbol.delete_node(valor, delete_opt.value)
    mostrar_mensaje(resultado)
    actualizar_arbol()

def on_search(b):
    valor = search_val.value.strip()
    if not valor:
        mostrar_mensaje("Ingrese un valor", error=True)
        return
    nodo = arbol.search_node(valor, search_opt.value)
    with output_info:
        clear_output()
        if nodo is None:
            display(widgets.HTML("<p style='color:red'>Nodo no encontrado</p>"))
        else:
            # Usamos las funciones de parentesco ya corregidas
            padre = BuscarPadre(arbol.raiz, nodo)
            abuelo = BuscarAbuelo(arbol.raiz, nodo)
            tio = BuscarTio(arbol.raiz, nodo)
            nivel = obtener_nivel(arbol.raiz, nodo)
            # Factor de balanceo (altura izquierda - altura derecha)
            def altura(n):
                return 1 + max(altura(n.izquierda), altura(n.derecha)) if n else 0
            fb = altura(nodo.izquierda) - altura(nodo.derecha)

            info = f"""
            <b>ID:</b> {nodo.data[0]}<br>
            <b>Título:</b> {nodo.data[1]}<br>
            <b>Satisfacción:</b> {nodo.satis:.5f}<br>
            <b>Rating:</b> {nodo.data[3]}<br>
            <b>Reseñas totales:</b> {nodo.data[4]}<br>
            <b>Positivas / Negativas / Neutrales:</b> {nodo.data[11]} / {nodo.data[12]} / {nodo.data[13]}<br>
            <b>Nivel:</b> {nivel}<br>
            <b>Factor balanceo:</b> {fb}<br>
            <b>Padre:</b> {padre.data[1] if padre else 'No tiene'}<br>
            <b>Abuelo:</b> {abuelo.data[1] if abuelo else 'No tiene'}<br>
            <b>Tío:</b> {tio.data[1] if tio else 'No tiene'}
            """
            display(widgets.HTML(info))

# Criterio a: positivas > negativas+neutras
def on_crit_a(b):
    resultados = []
    def recorrer(nodo):
        if nodo is None:
            return
        pos = float(nodo.data[11])
        neg = float(nodo.data[12])
        neu = float(nodo.data[13])
        if pos > (neg + neu):
            resultados.append(nodo.data[0])
        recorrer(nodo.izquierda)
        recorrer(nodo.derecha)
    recorrer(arbol.raiz)
    with output_info:
        clear_output()
        if resultados:
            display(widgets.HTML(f"<b>Cursos que cumplen:</b> {', '.join(resultados)}"))
        else:
            display(widgets.HTML("<p>No hay cursos que cumplan.</p>"))

# Criterio b: fecha de creación posterior a una fecha dada
def on_crit_b(b):
    fecha = date_picker.value
    if fecha is None:
        mostrar_mensaje("Seleccione una fecha", error=True)
        return
    resultados = []
    def recorrer(nodo):
        if nodo is None:
            return
        fecha_str = nodo.data[6]  # campo 'created'
        try:
            fecha_curso = datetime.datetime.strptime(fecha_str, "%Y-%m-%d").date()
            if fecha_curso > fecha:
                resultados.append(nodo.data[0])
        except:
            pass
        recorrer(nodo.izquierda)
        recorrer(nodo.derecha)
    recorrer(arbol.raiz)
    with output_info:
        clear_output()
        if resultados:
            display(widgets.HTML(f"<b>Cursos creados después de {fecha}:</b> {', '.join(resultados)}"))
        else:
            display(widgets.HTML(f"No hay cursos después de {fecha}."))

# Criterio c: número de clases en rango
def on_crit_c(b):
    rango = rango_clases.value
    resultados = []
    def recorrer(nodo):
        if nodo is None:
            return
        num_clases = float(nodo.data[5])  # num_published_lectures
        if rango[0] <= num_clases <= rango[1]:
            resultados.append(nodo.data[0])
        recorrer(nodo.izquierda)
        recorrer(nodo.derecha)
    recorrer(arbol.raiz)
    with output_info:
        clear_output()
        if resultados:
            display(widgets.HTML(f"<b>Cursos con clases entre {rango[0]} y {rango[1]}:</b> {', '.join(resultados)}"))
        else:
            display(widgets.HTML(f"No hay cursos en ese rango."))

# Criterio d: reseñas de un tipo > promedio de reseñas totales
def on_crit_d(b):
    tipo = tipo_resena.value
    total_reviews = 0
    count = 0
    def sumar_reviews(nodo):
        nonlocal total_reviews, count
        if nodo is None:
            return
        total_reviews += float(nodo.data[4])
        count += 1
        sumar_reviews(nodo.izquierda)
        sumar_reviews(nodo.derecha)
    sumar_reviews(arbol.raiz)
    if count == 0:
        mostrar_mensaje("Árbol vacío", error=True)
        return
    promedio = total_reviews / count
    resultados = []
    def filtrar(nodo):
        if nodo is None:
            return
        if tipo == "positive":
            valor = float(nodo.data[11])
        elif tipo == "negative":
            valor = float(nodo.data[12])
        else:
            valor = float(nodo.data[13])
        if valor > promedio:
            resultados.append(nodo.data[0])
        filtrar(nodo.izquierda)
        filtrar(nodo.derecha)
    filtrar(arbol.raiz)
    with output_info:
        clear_output()
        if resultados:
            display(widgets.HTML(f"<b>Cursos con {tipo} reviews > promedio ({promedio:.2f}):</b> {', '.join(resultados)}"))
        else:
            display(widgets.HTML(f"No hay cursos con {tipo} reviews mayores al promedio."))

# Recorrido por niveles recursivo
def on_niveles(b):
    if arbol.raiz is None:
        mostrar_mensaje("Árbol vacío", error=True)
        return
    def altura(n):
        return 1 + max(altura(n.izquierda), altura(n.derecha)) if n else 0
    h = altura(arbol.raiz)
    niveles = []
    for nivel in range(h):
        nivel_actual = []
        def recolectar(nodo, nivel_actual_n, nivel_objetivo):
            if nodo is None:
                return
            if nivel_actual_n == nivel_objetivo:
                nivel_actual.append(nodo.data[0])
            else:
                recolectar(nodo.izquierda, nivel_actual_n+1, nivel_objetivo)
                recolectar(nodo.derecha, nivel_actual_n+1, nivel_objetivo)
        recolectar(arbol.raiz, 0, nivel)
        niveles.append(nivel_actual)
    html = "<b>Recorrido por niveles (recursivo):</b><br>"
    for i, nivel in enumerate(niveles):
        html += f"Nivel {i}: {', '.join(nivel)}<br>"
    with output_info:
        clear_output()
        display(widgets.HTML(html))

# Conectar eventos
btn_insert.on_click(on_insert)
btn_delete.on_click(on_delete)
btn_search.on_click(on_search)
btn_crit_a.on_click(on_crit_a)
btn_crit_b.on_click(on_crit_b)
btn_crit_c.on_click(on_crit_c)
btn_crit_d.on_click(on_crit_d)
btn_niveles.on_click(on_niveles)

# Mostrar la interfaz
display(ui)
actualizar_arbol()  # muestra árbol vacío inicial